# Expert Profiling Demo

This notebook demonstrates how to profile individual experts in MoE layers to understand their specialization.

## What This Does

- **Force-routes** specific inputs to individual experts (bypassing the router)
- Tests each expert on **Math, Agentic, and Planning** tasks
- Generates a **skill matrix** showing what each expert learned
- Identifies **specialists** vs **generalists** vs **weak** experts

In [ ]:
import os
import sys
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer

# Add project root to path
sys.path.insert(0, os.path.abspath('../..'))

from mycelia.shared.config import ValidatorConfig
from mycelia.shared.dataloader import get_dataloader
from mycelia.shared.expert_profiler import (
    ExpertProfiler,
    print_skill_matrix,
    export_profiles_to_json,
)
from mycelia.shared.validation_integration import get_domain_specific_dataloader

print("Imports successful!")

## 1. Load Model and Config

In [ ]:
# Configuration
config = ValidatorConfig()
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

print(f"Using device: {device}")
print(f"Model path: {config.model.model_path}")

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(config.model.model_path)
print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")

In [ ]:
# Load model (may take a few minutes)
print("Loading model... (this may take a while)")
model = AutoModelForCausalLM.from_pretrained(
    config.model.model_path,
    torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
    device_map="auto" if device.type == "cuda" else None,
)

if device.type != "cuda":
    model = model.to(device)

model.eval()
print("Model loaded successfully!")
print(f"Model type: {model.__class__.__name__}")
print(f"Number of layers: {len(model.model.layers)}")

## 2. Prepare Domain-Specific Data

In [ ]:
# Create dataloaders for each expert group
# Note: You can use the validation_integration helpers

batch_size = 2
num_samples = 20  # Small number for quick testing

print("Creating domain-specific dataloaders...")

# Math dataloader (Expert Group 0)
math_config = ValidatorConfig()
math_config.task.data.batch_size = batch_size
math_config.task.expert_group_id = 0  # Math group
math_dataloader = get_dataloader(
    math_config,
    rank=0,
    world_size=1,
    tokenizer=tokenizer,
)

# Agentic dataloader (Expert Group 1)
agentic_config = ValidatorConfig()
agentic_config.task.data.batch_size = batch_size
agentic_config.task.expert_group_id = 1  # Agentic group
agentic_dataloader = get_dataloader(
    agentic_config,
    rank=0,
    world_size=1,
    tokenizer=tokenizer,
)

# Planning dataloader (Expert Group 2)
planning_config = ValidatorConfig()
planning_config.task.data.batch_size = batch_size
planning_config.task.expert_group_id = 2  # Planning group
planning_dataloader = get_dataloader(
    planning_config,
    rank=0,
    world_size=1,
    tokenizer=tokenizer,
)

print("Dataloaders created!")

## 3. Initialize Expert Profiler

In [ ]:
profiler = ExpertProfiler(
    model=model,
    tokenizer=tokenizer,
    device=device,
    num_samples_per_domain=num_samples,
)

print(f"Detected {len(profiler.moe_layers)} MoE layers: {profiler.moe_layers}")

## 4. Profile a Single Expert (Quick Test)

Let's test the profiling on one expert first.

In [ ]:
# Profile one expert as a test
test_layer_id = profiler.moe_layers[0]  # First MoE layer
test_expert_id = 0

print(f"\nProfiling Expert {test_expert_id} in Layer {test_layer_id}...\n")

expert_profile = profiler.profile_expert(
    layer_id=test_layer_id,
    expert_id=test_expert_id,
    math_dataloader=math_dataloader,
    agentic_dataloader=agentic_dataloader,
    planning_dataloader=planning_dataloader,
)

print("\n--- Expert Profile ---")
for key, value in expert_profile.to_dict().items():
    print(f"{key}: {value}")

## 5. Profile All Experts in One Layer

⚠️ **Warning**: This can take 10-30 minutes depending on:
- Number of experts in the layer
- Model size
- Number of samples
- Device speed

In [ ]:
# Profile one full layer
layer_id = profiler.moe_layers[0]

print(f"Profiling all experts in layer {layer_id}...")
print("This may take 10-30 minutes...\n")

layer_profile = profiler.profile_layer(
    layer_id=layer_id,
    math_dataloader=math_dataloader,
    agentic_dataloader=agentic_dataloader,
    planning_dataloader=planning_dataloader,
)

print("\n--- Layer Summary ---")
summary = layer_profile.get_summary()
for key, value in summary.items():
    print(f"{key}: {value}")

## 6. Visualize Expert Specializations

In [ ]:
# Create skill matrix heatmap
expert_ids = sorted(layer_profile.expert_profiles.keys())
domains = ['math_score', 'agentic_score', 'planning_score']

# Build matrix
skill_matrix = np.zeros((len(expert_ids), len(domains)))
for i, expert_id in enumerate(expert_ids):
    profile = layer_profile.expert_profiles[expert_id]
    skill_matrix[i, 0] = profile.math_score
    skill_matrix[i, 1] = profile.agentic_score
    skill_matrix[i, 2] = profile.planning_score

# Plot heatmap
plt.figure(figsize=(10, max(6, len(expert_ids) * 0.3)))
sns.heatmap(
    skill_matrix,
    annot=True,
    fmt=".2f",
    xticklabels=['Math', 'Agentic', 'Planning'],
    yticklabels=[f"Expert {eid}" for eid in expert_ids],
    cmap='YlOrRd',
    vmin=0,
    vmax=1,
    cbar_kws={'label': 'Score'},
)
plt.title(f'Expert Skill Matrix - Layer {layer_id}')
plt.xlabel('Domain')
plt.ylabel('Expert ID')
plt.tight_layout()
plt.show()

In [ ]:
# Specialization distribution pie chart
summary = layer_profile.get_summary()
spec_dist = summary['specialization_distribution']

plt.figure(figsize=(8, 8))
plt.pie(
    spec_dist.values(),
    labels=spec_dist.keys(),
    autopct='%1.1f%%',
    startangle=90,
)
plt.title(f'Specialization Distribution - Layer {layer_id}')
plt.show()

In [ ]:
# Bar chart comparing top experts per domain
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, (domain_name, score_attr) in enumerate([
    ('Math', 'math_score'),
    ('Agentic', 'agentic_score'),
    ('Planning', 'planning_score'),
]):
    experts = sorted(
        layer_profile.expert_profiles.values(),
        key=lambda ep: getattr(ep, score_attr),
        reverse=True,
    )[:10]  # Top 10
    
    expert_ids = [e.expert_id for e in experts]
    scores = [getattr(e, score_attr) for e in experts]
    colors = [{'math': 'blue', 'agentic': 'green', 'planning': 'orange', 'generalist': 'purple', 'weak': 'gray'}[e.specialization] for e in experts]
    
    axes[idx].barh(range(len(expert_ids)), scores, color=colors)
    axes[idx].set_yticks(range(len(expert_ids)))
    axes[idx].set_yticklabels([f"Expert {eid}" for eid in expert_ids])
    axes[idx].set_xlabel('Score')
    axes[idx].set_title(f'Top 10 {domain_name} Experts')
    axes[idx].set_xlim(0, 1)
    axes[idx].invert_yaxis()

plt.tight_layout()
plt.show()

## 7. Print Skill Matrix Report

In [ ]:
# Print formatted skill matrix
print_skill_matrix({layer_id: layer_profile}, top_k=5)

## 8. Export Results to JSON

In [ ]:
# Export to JSON file
output_path = f"expert_profiles_layer_{layer_id}.json"
export_profiles_to_json({layer_id: layer_profile}, output_path)
print(f"Profiles exported to: {output_path}")

## 9. (Optional) Profile ALL Layers

⚠️ **Warning**: This can take HOURS depending on your model!

Only run this if you have time and want a complete analysis.

In [ ]:
# Uncomment to run full profiling
# print("Starting FULL expert profiling (this will take hours)...")
# all_layer_profiles = profiler.profile_all_experts(
#     math_dataloader=math_dataloader,
#     agentic_dataloader=agentic_dataloader,
#     planning_dataloader=planning_dataloader,
# )

# print_skill_matrix(all_layer_profiles, top_k=5)
# export_profiles_to_json(all_layer_profiles, "expert_profiles_all_layers.json")
# print("Complete!")

## Analysis Tips

### Interpreting Results

1. **Specialists**: Experts with one domain score >0.8 and others <0.6
   - Good: Shows the model learned distinct specializations
   - Example: Math=0.92, Agentic=0.45, Planning=0.38

2. **Generalists**: All scores between 0.6-0.8
   - Neutral: Can handle multiple tasks but not exceptional at any
   - Example: Math=0.72, Agentic=0.68, Planning=0.71

3. **Weak Experts**: All scores <0.5
   - Bad: May indicate redundancy or poor training
   - Consider: Are these experts even being used by the router?

### Next Steps

1. **Compare with routing statistics**: Use `expert_selection.ipynb` to see which experts are actually selected
2. **Identify redundancy**: If many experts have similar profiles, some may be prunable
3. **Validate specialization**: Check if math specialists are routed to math problems
4. **Training insights**: Use this to understand if your expert groups are learning correctly